# Systematics 1D Ratio Test

This notebook tests the isolated `nplm_systematics` implementation of option A: finite-difference Falkon morphing for the nuisance ratio

$$
\log r(x;\nu) \simeq \nu\,\delta(x),\qquad
\delta(x) = \frac{g_+(x)-g_-(x)}{2\epsilon}.
$$

The toy is a truncated exponential with an analytic nuisance response, so the Falkon estimate can be compared directly to the true curve.

Use the `flk_torch113_cu116` kernel/environment for this notebook.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "examples":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from nplm_systematics import FalkonLogRatioEstimator, FiniteDifferenceMorpher
from nplm_systematics.morphing import LinearLogRMorphingCache
from nplm_systematics.toys import (
    sample_exponential_nuisance,
    truncated_exponential_log_ratio,
    truncated_exponential_log_ratio_derivative,
)

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})

## Configuration

The defaults are intentionally modest so the notebook runs quickly on CPU. Increase `n_central`, `n_varied`, `M`, and `iterations` for a smoother validation curve.

In [ ]:
seed = 123
rng = np.random.default_rng(seed)

n_central = 1000
n_varied = 1000
epsilon = 1.0

rate0 = 8.0
rate_log_step = 0.08
xmax = 1.0

ratio_config = {
    "sigma": None,
    "M": 100,
    "lambda": [1e-6],
    "iter": [30],
    "cg_tol": np.sqrt(1e-7),
    "keops": "no",
    "cpu": True,
    "seed": seed,
    "verbose": 0,
}

## Generate Central and Nuisance-Varied Samples

In [ ]:
x0 = sample_exponential_nuisance(
    n_central,
    0.0,
    rate0=rate0,
    rate_log_step=rate_log_step,
    xmax=xmax,
    rng=rng,
).reshape(-1, 1)

xp = sample_exponential_nuisance(
    n_varied,
    +epsilon,
    rate0=rate0,
    rate_log_step=rate_log_step,
    xmax=xmax,
    rng=rng,
).reshape(-1, 1)

xm = sample_exponential_nuisance(
    n_varied,
    -epsilon,
    rate0=rate0,
    rate_log_step=rate_log_step,
    xmax=xmax,
    rng=rng,
).reshape(-1, 1)

grid = np.linspace(0.0, xmax, 250, dtype=np.float64).reshape(-1, 1)
delta_true = truncated_exponential_log_ratio_derivative(
    grid.reshape(-1),
    rate0=rate0,
    rate_log_step=rate_log_step,
    xmax=xmax,
)

print(x0.shape, xp.shape, xm.shape)

## Analytic Derivative Sanity Check

Before involving Falkon, check that the closed-form derivative agrees with a small finite difference of the analytic log-ratio.

In [ ]:
eps_check = 1e-5
delta_fd = (
    truncated_exponential_log_ratio(
        grid.reshape(-1), eps_check, rate0=rate0, rate_log_step=rate_log_step, xmax=xmax
    )
    - truncated_exponential_log_ratio(
        grid.reshape(-1), -eps_check, rate0=rate0, rate_log_step=rate_log_step, xmax=xmax
    )
) / (2.0 * eps_check)

print("max |analytic derivative - finite difference| =", np.max(np.abs(delta_true - delta_fd)))

## Train the Finite-Difference Falkon Morpher

In [ ]:
if ratio_config["sigma"] is None:
    ratio_config["sigma"] = FalkonLogRatioEstimator.estimate_sigma_median(
        np.vstack([x0, xp, xm]),
        seed=seed,
    )

morpher = FiniteDifferenceMorpher(epsilon, ratio_config)
morpher.fit(x0, xp, xm)

g_plus, g_minus = morpher.predict_components(grid)
delta_falkon = morpher.predict_delta(grid)
residual = delta_falkon - delta_true

metrics = {
    "sigma": ratio_config["sigma"],
    "rmse": float(np.sqrt(np.mean(residual**2))),
    "mae": float(np.mean(np.abs(residual))),
    "corr": float(np.corrcoef(delta_falkon, delta_true)[0, 1]),
    "train_time_plus": morpher.summary_.train_time_plus,
    "train_time_minus": morpher.summary_.train_time_minus,
}
metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(grid.reshape(-1), delta_true, label="analytic $\\delta(x)$", lw=2)
axes[0].plot(grid.reshape(-1), delta_falkon, label="Falkon finite difference", lw=2)
axes[0].set_xlabel("x")
axes[0].set_ylabel("nuisance response")
axes[0].legend()

axes[1].plot(grid.reshape(-1), residual, lw=2)
axes[1].axhline(0.0, color="black", lw=1)
axes[1].set_xlabel("x")
axes[1].set_ylabel("Falkon - analytic")

fig.tight_layout()

## Check the Linearized Log-Ratio

For a test nuisance value, compare the linearized Falkon morpher against the exact analytic log-ratio. Differences include both Falkon estimation error and the first-order Taylor approximation.

In [ ]:
nu_test = 0.5
log_r_falkon = nu_test * delta_falkon
log_r_linear_true = nu_test * delta_true
log_r_exact = truncated_exponential_log_ratio(
    grid.reshape(-1),
    nu_test,
    rate0=rate0,
    rate_log_step=rate_log_step,
    xmax=xmax,
)

plt.plot(grid.reshape(-1), log_r_exact, label="exact $\\log r(x;\\nu)$", lw=2)
plt.plot(grid.reshape(-1), log_r_linear_true, label="analytic linearization", lw=2)
plt.plot(grid.reshape(-1), log_r_falkon, label="Falkon morpher", lw=2)
plt.xlabel("x")
plt.ylabel("log-ratio")
plt.legend();

## Cache Event-Level Responses

The profiled statistic only needs repeated evaluations on fixed reference/data events. Cache `delta_ref` and `delta_data` arrays instead of serializing Falkon models.

In [ ]:
x_data_like = sample_exponential_nuisance(
    250,
    0.25,
    rate0=rate0,
    rate_log_step=rate_log_step,
    xmax=xmax,
    rng=rng,
).reshape(-1, 1)

cache = morpher.build_cache(
    X_ref=x0[:500],
    X_data=x_data_like,
    metadata={"toy": "truncated_exponential_1d"},
)

cache_path = Path("/tmp") / "systematics_1d_ratio_cache.npz"
cache.save_npz(cache_path)
loaded = LinearLogRMorphingCache.load_npz(cache_path)

assert np.allclose(cache.delta_ref, loaded.delta_ref)
assert np.allclose(cache.delta_data, loaded.delta_data)
assert np.allclose(cache.log_r_ref(nu_test), loaded.log_r_ref(nu_test))

print("cache saved to", cache_path)
print("delta_ref shape:", loaded.delta_ref.shape)
print("delta_data shape:", loaded.delta_data.shape)
print("mean r_ref(nu_test):", float(np.mean(loaded.r_ref(nu_test))))